# 🍳 Optimizer Cookbook — Chapter 03: RMSprop

**Previous:** `02_adagrad.ipynb` — Adagrad  
**Next:** `04_adam.ipynb` — Adam

---

## Fixing Adagrad's Dying LR

Adagrad accumulates **all past squared gradients** — so $G_t$ grows without bound and the effective LR eventually dies.

**RMSprop** (Geoffrey Hinton, 2012 — unpublished, introduced in a Coursera lecture) replaces the cumulative sum with an **exponential moving average (EMA)**. This gives recent gradients more weight and lets old ones decay — so the effective LR stays alive throughout training.

---

## Update Rule

$$E[g^2]_t = \rho \cdot E[g^2]_{t-1} + (1 - \rho) \cdot (\nabla_{\theta} \mathcal{L})^2$$

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{E[g^2]_t + \epsilon}} \cdot \nabla_{\theta} \mathcal{L}$$

Where:
- $E[g^2]_t$ = exponential moving average of squared gradients
- $\rho$ = decay rate / smoothing factor (typically **0.99**)
- $\epsilon$ = numerical stability constant (default: `1e-8`)
- $\eta$ = learning rate (typically **0.001**)

**Key difference from Adagrad:** $E[g^2]_t$ is bounded — it never grows to infinity, so the LR never fully dies.

---

## RMSprop vs Adagrad — Intuition

| Property | Adagrad | RMSprop |
|---|---|---|
| Gradient accumulation | Cumulative sum | Exponential moving average |
| LR over time | Monotonically decreasing → 0 | Stabilises after warm-up |
| Memory of old gradients | Forever | Decays with factor ρ |
| Good for long training | ❌ | ✅ |

---

## RMSprop + Momentum

PyTorch's RMSprop supports an optional momentum term — combining adaptive LR with gradient acceleration:

$$v_t = \text{momentum} \cdot v_{t-1} + \frac{\eta}{\sqrt{E[g^2]_t + \epsilon}} \cdot \nabla \mathcal{L}$$
$$\theta_{t+1} = \theta_t - v_t$$

Enable with `momentum=0.9` in the optimizer.

---

## When to Use RMSprop

| Scenario | Verdict |
|---|---|
| RNNs / LSTMs (non-stationary loss surfaces) | ✅ Excellent — Hinton's original use case |
| Reinforcement Learning | ✅ Excellent (used in DQN, A3C) |
| CNNs when Adam is unstable | ✅ Good alternative |
| NLP Transformers | ❌ Adam/AdamW preferred |
| Sparse data / embeddings | ❌ Adagrad/Adam preferred |

---

## Key Hyperparameters

| Parameter | Default | Effect |
|---|---|---|
| `lr` | 0.01 | Step size — RMSprop is sensitive to this |
| `alpha` (ρ) | 0.99 | EMA decay — higher = smoother, slower adaptation |
| `eps` | 1e-8 | Numerical stability |
| `momentum` | 0 | Optional momentum term |
| `weight_decay` | 0 | L2 regularization |
| `centered` | False | If True, normalises gradient by its variance (more stable, slower) |

---
## 0. Imports & CUDA Check

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device : {device}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')

---
## 1. Config

In [ ]:
BATCH_SIZE     = 128
NUM_EPOCHS     = 20
NUM_CLASSES    = 10
NUM_WORKERS    = 2
SEED           = 42

# ── Optimizer Config ─────────────────────────────────────────
LEARNING_RATE  = 0.001
ALPHA          = 0.99      # EMA decay (ρ)
EPS            = 1e-8
MOMENTUM       = 0.0       # set to 0.9 to try RMSprop + Momentum
WEIGHT_DECAY   = 5e-4
CENTERED       = False
OPTIMIZER_NAME = 'RMSprop'

DATA_DIR    = '../data'
RESULTS_DIR = '../results/logs'
PLOTS_DIR   = '../results/plots'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Config loaded ✓')

---
## 2. Dataset — CIFAR-10

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_dataset = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

---
## 3. Model — SimpleCNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(SEED)
model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready. Trainable params: {total_params:,}')

---
## 4. Optimizer — RMSprop

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.RMSprop(
    model.parameters(),
    lr           = LEARNING_RATE,
    alpha        = ALPHA,
    eps          = EPS,
    momentum     = MOMENTUM,
    weight_decay = WEIGHT_DECAY,
    centered     = CENTERED
)

print(f'Optimizer    : RMSprop')
print(f'LR           : {LEARNING_RATE}')
print(f'Alpha (ρ)    : {ALPHA}')
print(f'Eps          : {EPS}')
print(f'Momentum     : {MOMENTUM}')
print(f'Centered     : {CENTERED}')
print(f'Weight Decay : {WEIGHT_DECAY}')
print('-' * 65)

---
## 5. Training Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

def run_training(model, train_loader, test_loader, optimizer, criterion, num_epochs, device, optimizer_name):
    history = []
    best_acc = 0.0
    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc     = evaluate(model, test_loader, criterion, device)
        elapsed = time.time() - t0
        if val_acc > best_acc: best_acc = val_acc
        history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                        'val_loss': val_loss, 'val_acc': val_acc, 'time_s': elapsed})
        print(f'Epoch [{epoch:02d}/{num_epochs}] Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | Time: {elapsed:.1f}s')
    print(f'\n✓ Best Val Accuracy: {best_acc:.2f}%')
    df = pd.DataFrame(history)
    df.to_csv(f'{RESULTS_DIR}/{optimizer_name}_log.csv', index=False)
    print(f'✓ Log saved → results/logs/{optimizer_name}_log.csv')
    return df

print('Utilities loaded ✓')

---
## 6. Train

In [ ]:
history = run_training(
    model          = model,
    train_loader   = train_loader,
    test_loader    = test_loader,
    optimizer      = optimizer,
    criterion      = criterion,
    num_epochs     = NUM_EPOCHS,
    device         = device,
    optimizer_name = OPTIMIZER_NAME
)

---
## 7. Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Training Curves — {OPTIMIZER_NAME}', fontsize=14, fontweight='bold')

ax1.plot(history['epoch'], history['train_loss'], label='Train Loss', linewidth=2, color='steelblue')
ax1.plot(history['epoch'], history['val_loss'],   label='Val Loss',   linewidth=2, color='tomato', linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history['epoch'], history['train_acc'], label='Train Acc', linewidth=2, color='steelblue')
ax2.plot(history['epoch'], history['val_acc'],   label='Val Acc',   linewidth=2, color='tomato', linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/{OPTIMIZER_NAME}_curves.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 8. Alpha (ρ) Sensitivity Sweep

Shows how the EMA decay rate affects convergence.  
Higher α = slower to forget old gradients = smoother but slower adaptation.

In [ ]:
alpha_values = [0.9, 0.95, 0.99, 0.999]
sweep_results = {}
SWEEP_EPOCHS = 5

for alpha in alpha_values:
    torch.manual_seed(SEED)
    m   = SimpleCNN(num_classes=NUM_CLASSES).to(device)
    opt = optim.RMSprop(m.parameters(), lr=LEARNING_RATE, alpha=alpha, eps=EPS)
    crit = nn.CrossEntropyLoss()
    val_accs = []
    for epoch in range(1, SWEEP_EPOCHS + 1):
        train_one_epoch(m, train_loader, crit, opt, device)
        _, val_acc = evaluate(m, test_loader, crit, device)
        val_accs.append(val_acc)
    sweep_results[f'α={alpha}'] = val_accs
    print(f'α={alpha} → Final Val Acc: {val_accs[-1]:.2f}%')

plt.figure(figsize=(9, 5))
for label, accs in sweep_results.items():
    plt.plot(range(1, SWEEP_EPOCHS + 1), accs, marker='o', linewidth=2, label=label)
plt.title('RMSprop — Alpha (ρ) Sensitivity', fontsize=13, fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Val Accuracy (%)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/RMSprop_alpha_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 9. Cumulative Comparison

In [ ]:
logs = {
    'Vanilla SGD'    : ('SGD_baseline_log.csv',  'gray',       '--'),
    'SGD + Momentum' : ('SGD_Momentum_log.csv',  'steelblue',  '--'),
    'Adagrad'        : ('Adagrad_log.csv',        'darkorange', '--'),
    'RMSprop'        : ('RMSprop_log.csv',        'mediumseagreen', '-'),
}

plt.figure(figsize=(10, 5))
for label, (fname, color, style) in logs.items():
    path = f'{RESULTS_DIR}/{fname}'
    if os.path.exists(path):
        df = pd.read_csv(path)
        plt.plot(df['epoch'], df['val_acc'], label=label, color=color, linestyle=style, linewidth=2)
    else:
        print(f'⚠️  Missing: {fname}')

plt.title('Val Accuracy — All Optimizers So Far', fontsize=13, fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Val Accuracy (%)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/comparison_up_to_rmsprop.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 10. Results Summary & Analysis

In [ ]:
best_epoch = history.loc[history['val_acc'].idxmax()]

print('=' * 55)
print(f'  {OPTIMIZER_NAME} — Summary')
print('=' * 55)
print(f"  Best Val Accuracy : {best_epoch['val_acc']:.2f}%")
print(f"  Best Val Loss     : {best_epoch['val_loss']:.4f}")
print(f"  Achieved at Epoch : {int(best_epoch['epoch'])}")
print(f"  Avg Time/Epoch    : {history['time_s'].mean():.1f}s")
print('=' * 55)
print()
print('📌 Key Observations:')
print('  ✅ LR stays alive throughout training — fixes Adagrads core flaw')
print('  ✅ Converges faster than vanilla SGD and often Adagrad on dense data')
print('  ✅ Great default for RNNs and RL problems')
print('  ⚠️  Can be noisy — adding momentum helps stabilise')
print('  ⚠️  Still slightly behind Adam on most modern benchmarks')
print('  ⚠️  Sensitive to LR — 0.001 is usually a safe starting point')
print()
print('📌 Real-world use cases:')
print('  → RNNs / LSTMs for sequence modelling')
print('  → Deep Q-Networks (DQN) and policy gradient RL')
print('  → When Adam is numerically unstable on your specific task')

---
## 11. What's Next?

RMSprop adapts the LR per parameter using a gradient EMA.  
**Adam** (Kingma & Ba, 2015) takes this further by also adding **momentum** (first moment estimation) — combining the best of SGD+Momentum and RMSprop into one optimizer.

Adam is the most widely used optimizer in modern deep learning. The next chapter is the most important one in the cookbook.

➡️ Continue to `04_adam.ipynb`